# Старт и проверка

Все данные синтетические. Работа выполняется в отдельной ветке; эталон v2 не изменяется.

Найдём корень проекта. В Colab при отсутствии файлов клонируется только рабочая ветка; локально используются уже полученные исходники.

In [ ]:
from pathlib import Path
import os, sys
root = Path.cwd()
while root != root.parent and not (root/'efc').exists():
    root = root.parent
if not (root/'efc').exists():
    import subprocess
    target = Path('/content/Forecast_Edge_Fog_Cloud')
    subprocess.run(['git','clone','--branch','night/fefc-v3','https://github.com/annfridreksen/Forecast_Edge_Fog_Cloud.git',str(target)],check=True)
    root=target
os.chdir(root)
sys.path.insert(0,str(root))
print('Рабочая папка:',root)


Установим зависимости только в Colab. В подготовленной локальной среде установка не требуется; сетевой доступ для тестов не нужен.

In [ ]:
import subprocess
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
print(sys.version)

Запустим тесты новой среды. Эталонные тесты v2 запускаются отдельно, без изменения их файлов.

In [ ]:
p=subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],text=True,capture_output=True)
print(p.stdout+p.stderr)
assert p.returncode==0
if (root/'v2/code').exists():
    p=subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=root/'v2/code',text=True,capture_output=True)
    print(p.stdout+p.stderr)
    assert p.returncode==0
else:
    print('Эталон v2 отсутствует в этой локальной копии; его проверка не выполнена.')
print('OK: доступные тесты')

Покажем устройство среды и движение одного вычисленного события. Это те же функции схем, что в тетради01, а не картинка без связи с моделью.

In [ ]:
from efc.views import architecture,animate_task
from efc import simulate
from efc.placement.study import controls
from IPython.display import HTML,display
import json,pandas as pd
architecture()
startup=simulate()
Path('results/startup_metrics.json').write_text(json.dumps(startup['metrics'],indent=2))
display(HTML(animate_task(startup['tasks'][0])))
control,_,_=controls()

Проверка. Числа выполненной ячейки сравниваются с первичным файлом результатов и заносятся в общий results/claims.json. Несовпадение вызывает ошибку, а не косметическое сообщение OK.

In [ ]:
from efc.claims import publish
records=[{'id':'startup_'+k,'value':startup['metrics'][k],'file':'results/startup_metrics.json','field':k} for k in ('mean_ms','p95_ms','mean_fog_wait_ms')]
records += [{'id':'startup_control_'+mode,'value':expected,'file':'results/placement_controls.csv','where':{'scenario':'minimal','requested_method':mode},'field':'time_ms'} for mode,expected in [('P',8),('B',4)]]
display(publish('00_Старт.ipynb',records))
print('OK: число в статье = число из модели')